# What makes a city healthy? A machine learning analysis of European cities
## Part 1 : Loading data 

## Data Sources

Data were sourced primarily from **Eurostat**, combining three geographic levels:
- **City-level** (Urban Audit): mortality, population structure, road deaths, employment rate, education level, cars per capita
- **NUTS2/NUTS3** (Nomenclature of Territorial Units for Statistics — regional administrative divisions): GDP per capita, population density, doctors per 100,000
- **Country-level**: obesity prevalence (sdg_02_10), smoking prevalence (sdg_03_30), unmet medical need (sdg_03_60)

Where city-level data was unavailable for specific countries, NUTS2 regional values were used as fallback for employment rate, car ownership and tertiary education rate.

**Road traffic and rail noise** exposure (% population exposed ≥55dB Lden/Lnight) was downloaded from the European Environment Agency Environmental Noise Directive 2017 reporting database (EEA END 2017).

**Air pollution** (PM2.5, NO2) city-level concentration estimates were sourced from Khomenko et al. (2021), *Lancet Planetary Health*, derived from validated spatial models calibrated against European air quality monitoring stations for 2015.

**Green space** coverage was sourced from Nieuwenhuijsen et al. (2021), *Lancet Planetary Health*.

**Swiss GDP** data unavailable in Eurostat was supplemented with NUTS3-level estimates from the ESPON Database.

In [37]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
import unicodedata

import sys
sys.path.append('Scripts')

from load_eurostat import load_file, merge_files, filter_cities, snapshot, load_country_file, load_nuts_file
from load_noise import load_noise
from load_isglobal import load_isglobal



## 1. Geographic reference table

A master lookup table is constructed linking each city to its corresponding NUTS3, NUTS2, NUTS1 and country codes. This enables subsequent merging of city-level Urban Audit data with regional (NUTS2/NUTS3) and country-level Eurostat indicators, as well as with the EEA noise data which uses a separate city naming convention.

In [2]:
# 1.1 making a master joining table for cities, regions and countries across eurostat and noise tables

# Load files
city_codes = pd.read_excel('DATA/Codes_city_NUTS_country/City_codes_noise_eurostat.xlsx')
lau = pd.read_excel('DATA/Codes_city_NUTS_country/CITY-LAU-2021.xlsx')

city_codes['City_code'] = city_codes['City_code'].str.strip()
lau['CITY CODE'] = lau['CITY CODE'].str.strip()
lau['NUTS 3 CODE'] = lau['NUTS 3 CODE'].str.strip()

# City -> NUTS3 (one row per city)
city_nuts = lau[['CITY CODE', 'NUTS 3 CODE']].drop_duplicates(subset=['CITY CODE'])
city_nuts.columns = ['City_code', 'NUTS3']
city_nuts['NUTS2'] = city_nuts['NUTS3'].str[:4]
city_nuts['NUTS1'] = city_nuts['NUTS3'].str[:3]
city_nuts['Country'] = city_nuts['NUTS3'].str[:2]

# Merge
result = city_codes.merge(city_nuts, on='City_code', how='left')
result = result[['City_code', 'city_eurostat', 'city_noise', 'NUTS3', 'NUTS2', 'NUTS1','Country']]

print(f'Shape: {result.shape}')
print(f'Cities with NUTS3: {result["NUTS3"].notna().sum()} / {len(result)}')
result.to_excel('DATA/Codes_city_NUTS_country/City_codes_full.xlsx', index=False)

Shape: (976, 7)
Cities with NUTS3: 732 / 976


Three cities have noise data (have city_noise matching), but are not present in the CITY-LAU-2021.xlsx file : Dabrowa Gornicza, Fuenlabrada and Parla. The last two are Madrid suburbs, I will add the corresponding NUTS3 (ES300) and NUTS2 (ES30) codes. Dąbrowa Górnicza — Polish city in Silesia, adding to dataframe NUTS3: PL22B, NUTS2: PL22

In [6]:
# 1.2 Manual fixes for cities missing from CITY-LAU-2021.xlsx
# Fuenlabrada and Parla -> Madrid suburbs (NUTS3: ES300, NUTS2: ES30)
# Dąbrowa Górnicza -> Polish city in Silesia (NUTS3: PL22B, NUTS2: PL22)

manual_fixes = {
    'Fuenlabrada':       {'NUTS3': 'ES300', 'NUTS2': 'ES30', 'Country': 'ES'},
    'Parla':             {'NUTS3': 'ES300', 'NUTS2': 'ES30', 'Country': 'ES'},
    'Dabrowa Gornicza':  {'NUTS3': 'PL22B', 'NUTS2': 'PL22', 'Country': 'PL'},
}

for city, codes in manual_fixes.items():
    mask = result['city_noise'] == city
    for col, val in codes.items():
        result.loc[mask, col] = val


In [7]:
# Renaming result to Codes
Codes = result

## 2. Load Eurostat Urban audit data

In [38]:
# 2.1 Load and merge city level eurostat data using a function merge_files from load_eurostat.py 

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

master = merge_files([
    'DATA/Eurostat/City/urb_cfermor$defaultview_page_total_65.xlsx',
    'DATA/Eurostat/City/urb_ctran__custom_21655731_spreadsheet.xlsx',
    'DATA/Eurostat/City/urb_cpop1__custom_21676206_spreadsheet_total_pop.xlsx',
    'DATA/Eurostat/City/urb_cpopstr__custom_21655848_spreadsheet_age_groups_prop.xlsx',
    'DATA/Eurostat/City/urb_clivcon__custom_21656079_spreadsheet.xlsx',
    'DATA/Eurostat/City/urb_clma$defaultview_page_spreadsheet.xlsx',
    'DATA/Eurostat/City/urb_percep__custom_21670138_spreadsheet.xlsx',
    'DATA/Eurostat/City/urb_ctran__custom_21800629_page_spreadsheet.xlsx',
    'DATA/Eurostat/City/urb_ceduc__custom_21914997_page_spreadsheet.xlsx',
])



Loading: urb_cfermor$defaultview_page_total_65.xlsx
  Sheet 1: [deaths_under65] — 825 cities, 2015-2024, 5911 values

Loading: urb_ctran__custom_21655731_spreadsheet.xlsx
  Sheet 1: [road_deaths_per10k] — 807 cities, 2015-2024, 5638 values

Loading: urb_cpop1__custom_21676206_spreadsheet_total_pop.xlsx
  Sheet 1: [pop_total] — 909 cities, 2016-2025, 6061 values
  Sheet 2: [pop_0_4] — 906 cities, 2016-2025, 5922 values
  Sheet 3: [pop_5_9] — 906 cities, 2016-2025, 5935 values
  Sheet 4: [pop_10_14] — 906 cities, 2016-2025, 6059 values
  Sheet 5: [pop_15_19] — 906 cities, 2016-2025, 6059 values
  Sheet 6: [pop_20_24] — 906 cities, 2016-2025, 6059 values
  Sheet 7: [pop_25_34] — 842 cities, 2016-2025, 2947 values
  Sheet 8: [pop_35_44] — 842 cities, 2016-2025, 2959 values
  Sheet 9: [pop_45_54] — 842 cities, 2016-2025, 2959 values
  Sheet 10: [pop_55_64] — 844 cities, 2016-2025, 3083 values
  Sheet 11: [pop_65_74] — 844 cities, 2016-2025, 2829 values
  Sheet 12: [pop_75_over] — 844 citie

In [9]:
# Verify the table set-up
master.sample(5)


,city,year,deaths_under65,road_deaths_per10k,pop_total,pop_0_4,pop_5_9,pop_10_14,pop_15_19,pop_20_24,...,healthcare_rather_unsat,healthcare_very_unsat,qol_decreased,qol_increased,health_bad,health_very_bad,lonely_all_time,lonely_most_time,cars_per1000,persons_aged_2564_with_isced_level_5_6_7
1328,Chartres (greater city),2024,129.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4714,Piacenza,2024,NaN,0.87,102887.0,3916.0,4469.0,4704.0,4697.0,4971.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,643.18,NaN
6281,Treviso,2023,83.0,0.00,84897.0,2904.0,3329.0,3665.0,3996.0,4120.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,625.46,15767.0
6391,Ulm,2022,NaN,0.08,126949.0,6015.0,5677.0,5599.0,5981.0,9729.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29000.0
3481,Logroño,2023,176.0,0.07,150206.0,5514.0,6973.0,8040.0,8040.0,7480.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,415.69,39090.0


In [10]:
print(master.columns)

Index(['city', 'year', 'deaths_under65', 'road_deaths_per10k', 'pop_total',
       'pop_0_4', 'pop_5_9', 'pop_10_14', 'pop_15_19', 'pop_20_24',
       'pop_25_34', 'pop_35_44', 'pop_45_54', 'pop_55_64', 'pop_65_74',
       'pop_75_over', 'median_age', 'pop_pct_65_74', 'pop_pct_75_over',
       'age_dependency', 'old_age_dependency', 'murders_violent',
       'median_income', 'one_person_hh_pct', 'lone_pensioner_hh_pct',
       'economically_active', 'healthcare_rather_unsat',
       'healthcare_very_unsat', 'qol_decreased', 'qol_increased', 'health_bad',
       'health_very_bad', 'lonely_all_time', 'lonely_most_time',
       'cars_per1000', 'persons_aged_2564_with_isced_level_5_6_7'],
      dtype='object')


In [11]:
# 2.2 Variable derivation
# Raw counts are converted to rates: premature mortality per 100,000 population under 65, employment rate (% of pop 20-64), and tertiary education rate (% of pop 25-64 with ISCED 5-8). 
# Missing population age values are forward-filled within each city before derivation. Intermediate columns are dropped after use.

# Step 0 - Forward fill population age groups where missing (e.g. Bulgaria 2019)
pop_cols = ['pop_25_34', 'pop_35_44', 'pop_45_54', 'pop_55_64',
            'pop_65_74', 'pop_75_over', 'pop_20_24']

master[pop_cols] = master.groupby('city')[pop_cols].ffill()


# Step 1 — derive calculated variables
master['pop_20_64']  = (master['pop_20_24'] + master['pop_25_34'] +
                        master['pop_35_44'] + master['pop_45_54'] +
                        master['pop_55_64'])
master['pop_65_over']         = master['pop_65_74'] + master['pop_75_over']
master['pop_under65']         = master['pop_total'] - master['pop_65_over']
master['pct_65_over']         = (master['pop_65_over'] / master['pop_total']) * 100
master['deaths_under65_rate'] = (master['deaths_under65'] / master['pop_under65']) * 100_000
master['employment_rate'] = (master['economically_active'] / master['pop_20_64']) * 100
master['murders_violent_per100k'] = (master['murders_violent'] / master['pop_total']) * 100_000
master['pct_poor_health']         = master['health_bad']              + master['health_very_bad']
master['pct_unhappy_healthcare']  = master['healthcare_rather_unsat'] + master['healthcare_very_unsat']
master['pop_25_64'] = (master['pop_25_34'] + master['pop_35_44'] +
                       master['pop_45_54'] + master['pop_55_64'])
master['tertiary_edu_pct'] = (master['persons_aged_2564_with_isced_level_5_6_7'] / master['pop_25_64']) * 100
# Step 2 — drop redundant/replaced columns
drop_cols = [
    # replaced by derived variables
    'deaths_under65',
    'health_bad', 'health_very_bad',
    'healthcare_rather_unsat', 'healthcare_very_unsat','murders_violent',
    'pop_0_4', 'pop_5_9', 'pop_10_14', 'pop_15_19',
    'pop_20_24', 'pop_25_34', 'pop_35_44', 'pop_45_54', 'pop_55_64',
    # redundant with pct_65_over
    'pop_pct_65_74', 'pop_pct_75_over',
    'age_dependency_ratio_population_aged_019',  # redundant with old_age_dependency,
    'lone_pensioner_hh_pct', # outside target age (<65)
    'persons_aged_2564_with_isced_level_5_6_7',  # replaced by tertiary_edu_pct
    'pop_25_64',          # intermediate only
]
master = master.drop(columns=drop_cols, errors='ignore')

In [12]:
print(master.columns)

Index(['city', 'year', 'road_deaths_per10k', 'pop_total', 'pop_65_74',
       'pop_75_over', 'median_age', 'age_dependency', 'old_age_dependency',
       'median_income', 'one_person_hh_pct', 'economically_active',
       'qol_decreased', 'qol_increased', 'lonely_all_time', 'lonely_most_time',
       'cars_per1000', 'pop_20_64', 'pop_65_over', 'pop_under65',
       'pct_65_over', 'deaths_under65_rate', 'employment_rate',
       'murders_violent_per100k', 'pct_poor_health', 'pct_unhappy_healthcare',
       'tertiary_edu_pct'],
      dtype='object')


In [13]:
# 2.3 Adding the city codes, NUTS3 and NUTS2, country to the master file

city_eurostat = Codes[["City_code","city_eurostat", 'NUTS3', 'NUTS2','NUTS1', 'Country']]
master_codes = master.merge(city_eurostat, left_on = "city", right_on='city_eurostat', how = "left")
master_codes = master_codes.drop(columns='city_eurostat')

In [14]:
# which cities miss city codes?

master_codes[master_codes['City_code'].isna()]['city'].unique()

array(['Middelburg', 'Reykjavík', 'Södertälje', 'Växjö'], dtype=object)

#### 'Middelburg', 'Reykjavík', 'Södertälje', 'Växjö' — these 4 cities do not have data in the Eurostat under-65 mortality table (primary variable) — missing City_codes can be ignored at this stage.

In [15]:
print(master_codes.columns)

Index(['city', 'year', 'road_deaths_per10k', 'pop_total', 'pop_65_74',
       'pop_75_over', 'median_age', 'age_dependency', 'old_age_dependency',
       'median_income', 'one_person_hh_pct', 'economically_active',
       'qol_decreased', 'qol_increased', 'lonely_all_time', 'lonely_most_time',
       'cars_per1000', 'pop_20_64', 'pop_65_over', 'pop_under65',
       'pct_65_over', 'deaths_under65_rate', 'employment_rate',
       'murders_violent_per100k', 'pct_poor_health', 'pct_unhappy_healthcare',
       'tertiary_edu_pct', 'City_code', 'NUTS3', 'NUTS2', 'NUTS1', 'Country'],
      dtype='object')


## 3. Load noise data

In [16]:
# 3.1 Load and merge city level noise data using a function load_noise from load_noise.py 

noise = load_noise(
    file_2017='DATA/Noise_data/END_2017.xlsx',
    file_2022='DATA/Noise_data/END_2022.xlsx'
)

Loading 2017 noise data...


C:\Users\rahel\Documents\Kursused\Data_analysis\FORMATION DATA ANALYST & IA- Mars 2026\FORMATION DATA ANALYST & IA- Mars 2026\MODULE 8 - Projet final\load_noise.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({


  Aggl_Road_Data (2017): 419 agglomerations (52 with no Lden data), Lden cols: ['55-59_Lden', '60-64_Lden', '65-69_Lden', '70-74_Lden', '>75_Lden'], Lnight cols: ['55-59_Lnight', '60-64_Lnight', '65-69_Lnight', '>70_Lnight']


C:\Users\rahel\Documents\Kursused\Data_analysis\FORMATION DATA ANALYST & IA- Mars 2026\FORMATION DATA ANALYST & IA- Mars 2026\MODULE 8 - Projet final\load_noise.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({


  Aggl_Rail_Data (2017): 419 agglomerations (53 with no Lden data), Lden cols: ['55-59_Lden_rail', '60-64_Lden_rail', '65-69_Lden_rail', '70-74_Lden_rail', '>75_Lden_rail'], Lnight cols: ['55-59_Lnight_rail', '60-64_Lnight_rail', '65-69_Lnight_rail', '>70_Lnight_rail']

Loading 2022 noise data...


C:\Users\rahel\Documents\Kursused\Data_analysis\FORMATION DATA ANALYST & IA- Mars 2026\FORMATION DATA ANALYST & IA- Mars 2026\MODULE 8 - Projet final\load_noise.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({


  Aggl_Road_Data (2022): 415 agglomerations (102 with no Lden data), Lden cols: ['55-59_Lden', '60-64_Lden', '65-69_Lden', '70-74_Lden', '>75_Lden'], Lnight cols: ['55-59    _Lnight', '60-64    _Lnight', '65-69    _Lnight', '>70_Lnight']


C:\Users\rahel\Documents\Kursused\Data_analysis\FORMATION DATA ANALYST & IA- Mars 2026\FORMATION DATA ANALYST & IA- Mars 2026\MODULE 8 - Projet final\load_noise.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({


  Aggl_Rail_Data (2022): 415 agglomerations (128 with no Lden data), Lden cols: ['55-59_Lden_rail', '60-64_Lden_rail', '65-69_Lden_rail', '70-74_Lden_rail', '>75_Lden_rail'], Lnight cols: ['55-59    _Lnight_rail', '60-64    _Lnight_rail', '65-69    _Lnight_rail', '>70_Lnight_rail']

Noise dataset: 834 rows, 447 unique cities, years: [np.int64(2017), np.int64(2022)]


In [17]:
# 3.2 Adding City_code to the noise data

# City names in the END noise file use EEA naming conventions which differ from Eurostat Urban Audit names. A manual mapping table (city_noise → City_code) 
# is used to match noise agglomerations to their corresponding Eurostat city codes. Unmatched cities are printed for inspection — these are typically 
# cities outside the study scope (Turkey, UK) or agglomerations not covered by Urban Audit.

noise['city'] = noise['city'].str.strip()
city_noise = Codes[["City_code","city_noise"]]

noise_codes =  noise.merge(city_noise, left_on='city', right_on='city_noise', how='left')
# Check what's still unmatched
unmatched = noise_codes[noise_codes['City_code'].isna()]['city'].unique()
print(f'Still unmatched: {len(unmatched)}')
print(sorted(unmatched))

Still unmatched: 62
['Akureyri', 'Alcobendas', 'Alcorcon', 'Amadora', 'Baden-Brugg', 'Baix Llobregat I', 'Baix Llobregat II', 'Baix Llobregat Nord', 'Baix Llobregat Sud', 'Barakaldo', 'Barcelona', 'Barcelones Nord', 'Besancon', 'Chorzow', 'Drammen', 'Drobeta-Turnu Severin', 'Eskilstuna', 'Fredrikstad/Sarpsborg', 'Fribourg', 'Gardabaer', 'Gavle', 'Getafe', 'Hafnafjordur', 'Halmstad', 'Hanau', 'Henin-Beaumont', 'Hildesheim', 'Huddinge', 'Kauniainen', 'Kopavogur', "L'Hospitalet de Llobregat", 'Larnaca', 'Leganes', 'Leuven', 'Limassol', 'Limerick', 'Limoges', 'Luxembourg_south', 'Matosinhos', 'Metz', 'Mosfellsbaer', 'Mostoles', 'Nacka', 'Odivelas', 'Oeiras', 'Pafos', 'Piatra-Neamt', 'Piraeus', 'Poitiers', 'Ramnicu Valcea', 'Reykjanesbaer', 'Reykjavik', 'Roquetas de Mar', 'San Cristobal de La Laguna', 'Selfoss', 'Seltjarnarnes', 'Suceava', 'Valles Occidental I', 'Valles Occidental II', 'Vaslui', 'Versailles', 'Zug']


In [18]:
# 3.3 Aggregating noise to city level

# Large cities (e.g. Paris) are split into multiple sub-agglomerations in the END database but reported as a single greater city in Eurostat — groupby average reconciles this.
# Road and rail noise are combined into composite day (Lden) and night (Lnight) measures.

noise_grouped = noise_codes.groupby(['City_code', 'year'])[
    ['road_lden_pct_55', 'road_lnight_pct_55', 
     'rail_lden_pct_55', 'rail_lnight_pct_55']
].mean().reset_index()

# Composite noise variables — road + rail combined
# Night noise has stronger health effects than daytime (sleep disruption)
noise_grouped['total_lnight_pct_55'] = (noise_grouped['road_lnight_pct_55'] + 
                                         noise_grouped['rail_lnight_pct_55'])
noise_grouped['total_lden_pct_55'] = (noise_grouped['road_lden_pct_55'] + 
                                       noise_grouped['rail_lden_pct_55'])

# Drop separate road/rail columns — replaced by composites
noise_grouped = noise_grouped.drop(columns=['road_lden_pct_55', 'road_lnight_pct_55',
                                             'rail_lden_pct_55', 'rail_lnight_pct_55'])

In [19]:
# 3.4 Merging the noise data to the master file

master_noise = master_codes.merge(noise_grouped, on=['City_code','year'], how='left')
master_noise.columns

Index(['city', 'year', 'road_deaths_per10k', 'pop_total', 'pop_65_74',
       'pop_75_over', 'median_age', 'age_dependency', 'old_age_dependency',
       'median_income', 'one_person_hh_pct', 'economically_active',
       'qol_decreased', 'qol_increased', 'lonely_all_time', 'lonely_most_time',
       'cars_per1000', 'pop_20_64', 'pop_65_over', 'pop_under65',
       'pct_65_over', 'deaths_under65_rate', 'employment_rate',
       'murders_violent_per100k', 'pct_poor_health', 'pct_unhappy_healthcare',
       'tertiary_edu_pct', 'City_code', 'NUTS3', 'NUTS2', 'NUTS1', 'Country',
       'total_lnight_pct_55', 'total_lden_pct_55'],
      dtype='object')

In [20]:
master_noise.sample(5)

,city,year,road_deaths_per10k,pop_total,pop_65_74,pop_75_over,median_age,age_dependency,old_age_dependency,median_income,...,pct_poor_health,pct_unhappy_healthcare,tertiary_edu_pct,City_code,NUTS3,NUTS2,NUTS1,Country,total_lnight_pct_55,total_lden_pct_55
4643,Paris (greater city),2019,0.16,10277625.0,809800.0,699771.0,36.0,66.7,24.5,23950.0,...,4.7,19.6,49.522041,FR001C,FR101,FR10,FR1,FR,NaN,NaN
6317,Toruń,2020,0.45,198801.0,23604.0,14865.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,PL013C,PL613,PL61,PL6,PL,NaN,NaN
6964,Wrexham,2015,0.44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,UK021C,NaN,NaN,NaN,NaN,NaN,NaN
2220,Gera,2020,0.21,93125.0,12281.0,14681.0,52.5,NaN,NaN,NaN,...,NaN,NaN,NaN,DE113C,DEG02,DEG0,DEG,DE,NaN,NaN
5544,San Vicente del Raspeig/Sant Vicent del Raspeig,2018,0.17,57785.0,4579.0,3517.0,40.3,57.3,22.0,NaN,...,NaN,NaN,34.239227,ES098C,ES521,ES52,ES5,ES,NaN,NaN


## 4. Load air pollution and green space data

In [21]:
# 4.1 Load and merge city level air pollution and green space data using a function load_isglobal from load_isglobal.py 
env = load_isglobal(
    air_file        = 'DATA/Other_sources/City_air_pol.xlsx',
    ndvi_file       = 'DATA/Other_sources/City_green_NDVI.xlsx',
    ga_file         = 'DATA/Other_sources/City_green_GA_perc.xlsx',
)

Air pollution: 857 cities
NDVI:          866 cities
Green area %:  866 cities

Merged ISGlobal dataset: 866 cities
PM2.5 coverage:  857
NO2 coverage:    857
NDVI coverage:   866
Green area %:    866


In [22]:
env.sample(5)

,city_code,city,year,pm25_median,no2_median,ndvi_mean,ga_pct_mean
756,UK004K1,Greater Glasgow,2015,7.365500,21.343210,0.580468,33.559319
535,IT507C1,Ferrara,2015,20.844070,20.889950,0.543126,69.705070
767,UK016C1,Aberdeen,2015,6.479024,14.458545,0.563164,37.273167
412,FR304C1,Melun,2015,12.906825,20.596690,0.595345,36.822174
285,ES525C1,Tarragona,2015,15.024445,24.642520,0.351128,42.323071


In [23]:
# City codes in ISGlobal files differ from Eurostat Urban Audit codes — joining by city name instead
env = env.drop(columns=['city_code'])

In [24]:
# 4.2 City name normalization
# ISGlobal and Eurostat use different naming conventions — accents, slashes, suffixes and Dutch "Greater X" prefixes all cause mismatches. 
# A normalize_city() function standardizes both sides before joining.

def normalize_city(s):
    s = str(s).strip()
    # Remove (greater city) suffix
    s = s.replace(' (greater city)', '')
    # Take only part before / (handles Turku/Åbo, Bruxelles/Brussel, Espoo/Esbo etc.)
    s = s.split('/')[0].strip()
    # Manual fix for Polish Ł/ł (not handled by NFKD)
    s = s.replace('Ł', 'L').replace('ł', 'l')
    # Fix ø/Ø (Danish/Norwegian - drops entirely in NFKD)
    s = s.replace('ø', 'o').replace('Ø', 'O')
    # Normalize to ASCII (handles all other accents)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii').strip()
    return s

# Name corrections applied to both sides after normalization
manual_map = {
    'Geneve':     'Geneva',
    'St, Gallen': 'St. Gallen',
    'Annemasse-Geneva, FR part':       'Annemasse',
}

# Dutch Greater X mapping
greater_city_map = {
    "Amsterdam":      "Greater Amsterdam",
    "Rotterdam":      "Greater Rotterdam",
    "s-Gravenhage":   "Greater s-Gravenhage",
    "Eindhoven":      "Greater Eindhoven",
    "Utrecht":        "Greater Utrecht",
    "Haarlem":        "Greater Haarlem",
    "Leiden":         "Greater Leiden",
    "Heerlen":        "Greater Heerlen",
    "Alkmaar":        "Greater Alkmaar",
    "Nissewaard":     "Greater Nissewaard",
    "Sittard-Geleen": "Greater Sittard-Geleen",
    "Heemskerk":      "Greater Heemskerk",
    "Soest":          "Greater Soest",
    "Middelburg":     "Greater Middelburg",
    "Ede":            "Greater Ede",
}

# Apply to master_noise
master_noise['city_clean'] = (master_noise['city']
    .apply(normalize_city)
    .replace(manual_map)
    .replace(greater_city_map))

# Apply to env
env['city_clean'] = (env['city']
    .apply(normalize_city)
    .replace(manual_map)
    .replace(greater_city_map))




In [25]:
# 4.3 Merging air pollution and green space data to the master 

master_noise_pol = master_noise.merge(
    env.drop(columns=['city','year']),
    on='city_clean', how='left')

In [26]:
master_noise_pol.sample(5)

,city,year,road_deaths_per10k,pop_total,pop_65_74,pop_75_over,median_age,age_dependency,old_age_dependency,median_income,...,NUTS2,NUTS1,Country,total_lnight_pct_55,total_lden_pct_55,city_clean,pm25_median,no2_median,ndvi_mean,ga_pct_mean
1686,Dos Hermanas,2019,0.15,133968.0,10246.0,6652.0,40.4,NaN,NaN,NaN,...,ES61,ES6,ES,NaN,NaN,Dos Hermanas,13.03359,19.43502,0.280620,37.119625
2799,Italy,2019,0.53,59816673.0,6656851.0,6987512.0,45.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Italy,NaN,NaN,NaN,NaN
4578,Paredes (greater city),2022,0.35,84972.0,7544.0,5219.0,43.0,NaN,NaN,NaN,...,PT11,PT1,PT,NaN,NaN,Paredes,12.28270,18.18391,0.548142,63.846455
990,Brindisi,2016,0.34,88302.0,10238.0,8565.0,43.0,67.4,35.6,NaN,...,ITF4,ITF,IT,NaN,NaN,Brindisi,15.11057,15.15072,0.380416,51.987514
4024,Mérida,2018,0.34,59352.0,4893.0,4260.0,41.3,59.7,24.6,NaN,...,ES43,ES4,ES,NaN,NaN,Merida,10.45996,13.26986,0.308346,57.174484


In [27]:
# 4.4 Check how many cities have air quality data

total = master_noise_pol['city'].nunique()
matched = master_noise_pol[master_noise_pol['pm25_median'].notna()]['city'].nunique()
print(f'Cities with air quality data: {matched} / {total} ({matched/total*100:.0f}%)')


Cities with air quality data: 760 / 920 (83%)


In [28]:
master_noise_pol.columns

Index(['city', 'year', 'road_deaths_per10k', 'pop_total', 'pop_65_74',
       'pop_75_over', 'median_age', 'age_dependency', 'old_age_dependency',
       'median_income', 'one_person_hh_pct', 'economically_active',
       'qol_decreased', 'qol_increased', 'lonely_all_time', 'lonely_most_time',
       'cars_per1000', 'pop_20_64', 'pop_65_over', 'pop_under65',
       'pct_65_over', 'deaths_under65_rate', 'employment_rate',
       'murders_violent_per100k', 'pct_poor_health', 'pct_unhappy_healthcare',
       'tertiary_edu_pct', 'City_code', 'NUTS3', 'NUTS2', 'NUTS1', 'Country',
       'total_lnight_pct_55', 'total_lden_pct_55', 'city_clean', 'pm25_median',
       'no2_median', 'ndvi_mean', 'ga_pct_mean'],
      dtype='object')

## 5. Loading regional and country-level indicators

Socioeconomic, demographic and healthcare variables unavailable at city level are loaded at NUTS2, NUTS3 and country level. NUTS2 regional values serve as fallback for countries with incomplete Urban Audit reporting for employment, car ownership and education. Swiss GDP per capita (unavailable in Eurostat) is sourced from the ESPON database.

In [29]:
# 5.1 Loading variables using functions from load_eurostat.py

# Country level (2-char codes)
obesity = load_country_file('DATA/Eurostat/sdg_02_10_page_spreadsheet.xlsx', 'obesity_pct')
smoking = load_country_file('DATA/Eurostat/sdg_03_30_page_spreadsheet.xlsx', 'smoking_pct')
unmet   = load_country_file('DATA/Eurostat/sdg_03_60_page_spreadsheet.xlsx', 'unmet_medical_pct')

# NUTS2 (4-char codes)
beds    = load_nuts_file('DATA/Eurostat/tgs00064_page_spreadsheet.xlsx', 'hospital_beds_per100k', nuts_level=2)
doctors_n2 = load_nuts_file('DATA/Eurostat/hlth_rs_prsrg__custom_21799569_spreadsheet.xlsx', 'doctors_n2', nuts_level=2)
# Doctors per population is mainly at NUTS2 level, with an aberation for Germany - on NUTS1 level
# Load doctors data at NUTS1 (covers Germany)
doctors_n1 = load_nuts_file('DATA/Eurostat/hlth_rs_prsrg__custom_21799569_spreadsheet.xlsx','doctors_n1', nuts_level=1)
pop_nuts2 = load_nuts_file('DATA/Eurostat/tgs00096_page_spreadsheet.xlsx', 'pop_nuts2', nuts_level=2)

# NUTS3 (5-char codes)
gdp  = load_nuts_file('DATA/Eurostat/nama_10r_3gdp__custom_21799900_spreadsheet.xlsx', 'gdp_pps', nuts_level=3)
dens = load_nuts_file('DATA/Eurostat/demo_r_d3dens__custom_21799742_page_spreadsheet.xlsx', 'pop_density', nuts_level=3)

# NUTS1 (3-char codes)
# doctors count for Germany is in the above NUTS2 file actually at NUTS1 level - loading NUTS1 population for normalization
pop_nuts1 =  load_nuts_file('DATA/Eurostat/demo_r_pjanaggr3__custom_21821182_page_spreadsheet.xlsx','pop_nuts1', nuts_level=1)

# City-level employment rate has been used where available, but NUTS2 regional rate used as proxy for Romania, Poland, Portugal and Slovakia where city-level data was not reported in Urban Audit
emp_nuts2 = load_nuts_file('DATA/Eurostat/lfst_r_lfe2emprt__custom_21890367_spreadsheet.xlsx','emp_nuts2', nuts_level=2)

# City-level cars per 1000 habitants has been used where available, but NUTS2 regional data used as proxy for Romania, Bulgaria, Portugal and Czech where city-level data was not reported in Urban Audit
cars_nuts2 = load_nuts_file('DATA/Eurostat/tran_r_vehst__custom_21890727_page_spreadsheet.xlsx','cars_count_nuts2', nuts_level=2)

# Tertiary education rate — city-level from Urban Audit where available, NUTS2 regional rate used as proxy for BE, CZ, NL, PL, PT, RO, SK where city-level data was not reported
edu_nuts2 = load_nuts_file('DATA/Eurostat/edat_lfse_04__custom_21915116_page_spreadsheet.xlsx', 'tertiary_edu_nuts2', nuts_level=2)

## gpd purchasing power standards for Switzerland are missing in eurostat, using NUTS3 level data from https://database.espon.eu/indicator/320/ 
ch_gdp = pd.read_excel('DATA/Other_sources/Espon_CH_gpd_pps.xlsx')

# Melt to long format
ch_gdp_long = ch_gdp.melt(id_vars=['NUTS3', 'City'], 
    value_vars=[2015, 2016, 2017, 2018],
    var_name='year', 
    value_name='gdp_pps_ch'
)
ch_gdp_long['year'] = ch_gdp_long['year'].astype(int)

obesity_pct: 37 countries, years [np.int64(2008), np.int64(2014), np.int64(2017), np.int64(2019), np.int64(2022), np.int64(2025)], 178 values
smoking_pct: 29 countries, years [np.int64(2006), np.int64(2009), np.int64(2012), np.int64(2014), np.int64(2017), np.int64(2020), np.int64(2023)], 200 values
unmet_medical_pct: 38 countries, years [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)], 382 values
hospital_beds_per100k: 227 NUTS2 regions, years [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)], 2033 values
doctors_n2: 285 NUTS2 regions, years [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)], 1486 values
doctors_n1: 16 NUTS1 regions, years [np.int64(2015), np.int64(2016), np.int64(2017), np

In [30]:
# 5.2 Merging the new variable to the master file

# Merge into master_noise_pol
master_noise_pol = master_noise_pol.merge(obesity, on=['Country','year'], how='left')
master_noise_pol = master_noise_pol.merge(smoking, on=['Country','year'], how='left')

# Switzerland not in Eurostat SDG smoking data — patch with 2017 value (27%) from
# https://www.bfs.admin.ch/bfs/fr/home/statistiques/sante/enquetes/sgb/resultats-publications.assetdetail.28725086.html
master_noise_pol.loc[master_noise_pol['Country'] == 'CH', 'smoking_pct'] = 27.0
# Netherlands smoking — Trimbos Institute (National Drug Monitor) ;  Annual values: 2015: 26.3%, 2016: 24.1%, 2017: 23.1%, 2018: 22.4%, 2019: 21.7%
# https://www.trimbos.nl/wp-content/uploads/2025/10/AF1999-Smoking-in-the-Netherlands-Key-statistics-for-2021.pdf
nl_smoking_avg = round((26.3 + 24.1 + 23.1 + 22.4 + 21.7) / 5, 0)
master_noise_pol.loc[master_noise_pol['Country'] == 'NL', 'smoking_pct'] = nl_smoking_avg

master_noise_pol = master_noise_pol.merge(unmet,   on=['Country','year'], how='left')
master_noise_pol = master_noise_pol.merge(beds,    on=['NUTS2','year'], how='left')
master_noise_pol = master_noise_pol.merge(doctors_n2, on=['NUTS2','year'], how='left')
master_noise_pol = master_noise_pol.merge(doctors_n1, on=['NUTS1','year'], how='left')
master_noise_pol = master_noise_pol.merge(pop_nuts2, on=['NUTS2','year'], how='left')
master_noise_pol = master_noise_pol.merge(gdp,     on=['NUTS3','year'], how='left')
master_noise_pol = master_noise_pol.merge(dens,    on=['NUTS3','year'], how='left')
master_noise_pol = master_noise_pol.merge(pop_nuts1, on=['NUTS1','year'], how='left')

master_noise_pol = master_noise_pol.merge(emp_nuts2, on=['NUTS2','year'], how='left')
master_noise_pol = master_noise_pol.merge(cars_nuts2, on=['NUTS2','year'], how='left')
master_noise_pol = master_noise_pol.merge(edu_nuts2, on=['NUTS2', 'year'], how='left')
master_noise_pol = master_noise_pol.merge(ch_gdp_long[['NUTS3', 'year', 'gdp_pps_ch']], on=['NUTS3', 'year'], how='left')



In [31]:
# 5.3 Normalizing and filling gaps in regional indicators
# Doctors per 100,000 is calculated from raw counts divided by NUTS2 population, with Germany using NUTS1-level data (doctors reported at Länder level in Eurostat).
# Employment rate, car ownership and tertiary education are filled with NUTS2 regional values where city-level Urban Audit data is missing.
# Swiss GDP is filled from ESPON NUTS3 estimates.

# Compute doctors_per100k: NUTS2-based for everyone, NUTS1-based override for Germany
master_noise_pol['doctors_per100k'] = (
    master_noise_pol['doctors_n2'] / master_noise_pol['pop_nuts2'] * 100_000
)

mask_de = master_noise_pol['Country'] == 'DE'
master_noise_pol.loc[mask_de, 'doctors_per100k'] = (
    master_noise_pol.loc[mask_de, 'doctors_n1'] /
    master_noise_pol.loc[mask_de, 'pop_nuts1'] * 100_000
)

# Fill missing city-level employment with NUTS2 value
master_noise_pol['employment_rate'] = master_noise_pol['employment_rate'].fillna(
    master_noise_pol['emp_nuts2'])

# Normalize car numbers per nuts2 to per 1000 inhabitants using pop_nuts2
master_noise_pol['cars_nuts2_per1000'] = (
    master_noise_pol['cars_count_nuts2'] / 
    master_noise_pol['pop_nuts2'] * 1000
)

# Fill missing city-level cars_per1000 with NUTS2 value
master_noise_pol['cars_per1000'] = master_noise_pol['cars_per1000'].fillna(
    master_noise_pol['cars_nuts2_per1000'])

# Fill missing city-level education with NUTS2 value
master_noise_pol['tertiary_edu_pct'] = master_noise_pol['tertiary_edu_pct'].fillna(
    master_noise_pol['tertiary_edu_nuts2'])

# Fill missing gdp_pps for Switzerland
master_noise_pol['gdp_pps'] = master_noise_pol['gdp_pps'].fillna(
    master_noise_pol['gdp_pps_ch'])

## 6. Removing low coverage or redundant variables

In [35]:
# 6.1 Coverage check on current master_noise_pol

mort = master_noise_pol[
    (master_noise_pol['year'].isin([2015, 2016, 2017, 2018, 2019])) &
    (master_noise_pol['deaths_under65_rate'].notna())
]
n_cities = mort['City_code'].nunique()
print(f'Cities with mortality data 2015-2019: {n_cities}')
print()
print('Coverage per variable (% of those cities):')
print('-'*55)

numeric_cols = [c for c in mort.select_dtypes(include='number').columns 
                if c != 'year']

coverage = {}
for var in numeric_cols:
    n = mort.groupby('City_code')[var].apply(lambda x: x.notna().any()).sum()
    coverage[var] = n

coverage_df = pd.DataFrame.from_dict(coverage, orient='index', columns=['n_cities'])
coverage_df['pct'] = (coverage_df['n_cities'] / n_cities * 100).round(0).astype(int)
coverage_df = coverage_df.sort_values('n_cities', ascending=False)

for var, row in coverage_df.iterrows():
    print(f'{var:<30} {row["n_cities"]:>4} / {n_cities}  ({row["pct"]}%)')

Cities with mortality data 2015-2019: 788

Coverage per variable (% of those cities):
-------------------------------------------------------
pop_total                       788 / 788  (100%)
old_age_dependency              788 / 788  (100%)
deaths_under65_rate             788 / 788  (100%)
employment_rate                 783 / 788  (99%)
tertiary_edu_pct                779 / 788  (99%)
cars_per1000                    769 / 788  (98%)
ga_pct_mean                     705 / 788  (89%)
road_deaths_per10k              700 / 788  (89%)
no2_median                      698 / 788  (89%)
pm25_median                     698 / 788  (89%)
unmet_medical_pct               632 / 788  (80%)
obesity_pct                     625 / 788  (79%)
pop_density                     625 / 788  (79%)
doctors_per100k                 624 / 788  (79%)
smoking_pct                     622 / 788  (79%)
gdp_pps                         598 / 788  (76%)
total_lden_pct_55               277 / 788  (35%)
total_lnight_pct_55   

In [33]:
# 6.2 Removing low coverage and redundant variables

master_noise_pol = master_noise_pol.drop(columns=[
    # --- Too sparse, unusable ---
    'median_income',               # 24% coverage, replaced by gdp_pps
    'pct_poor_health',             # 7% coverage, unusable
    'pct_unhappy_healthcare',      # 9% coverage, unusable
    'qol_decreased',               # 7% coverage, unusable
    'qol_increased',               # 7% coverage, unusable
    'lonely_all_time',             # 0% coverage, unusable
    'lonely_most_time',            # 0% coverage, unusable
    

    # --- Redundant variables ---
    'age_dependency',              # redundant with old_age_dependency
    'median_age',                  # redundant with old_age_dependency (same age structure signal)
    'pct_65_over',                 # redundant with old_age_dependency
    'ndvi_mean',                   # redundant with ga_pct_mean (both measure green space)

    # --- Intermediate derivation columns, no longer needed ---
    'economically_active',         # used to derive employment_rate, raw count
    'pop_65_74',                   # used to derive pop_65_over
    'pop_75_over',                 # used to derive pop_65_over
    'pop_65_over',                 # used to derive pop_under65 and pct_65_over
    'pop_under65',                 # used to derive deaths_under65_rate
    'pop_20_64',                   # used to derive employment_rate
    'doctors_n1',                  # intermediate: raw doctor count at NUTS1 (Germany)
    'doctors_n2',                  # intermediate: raw doctor count at NUTS2
    'pop_nuts1',                   # intermediate: used to normalize doctors for Germany
    'pop_nuts2',                   # intermediate: used to normalize doctors_per100k
    'emp_nuts2',                   # intermediate: employment rate at NUTS2 (CZ,PL,RO,SK)
    'cars_count_nuts2',            # intermediate: car count at NUTS2 (BG,CZ,PT,RO)
    'cars_nuts2_per1000',          # intermediate: car ownership rate at NUTS2 (BG,CZ,PT,RO)
    'gdp_pps_ch',                  # intermediate: Swiss gdp_pps
    'tertiary_edu_nuts2',          # intermediate: education at NUTS2

    # --- Replaced by better variable ---
    'hospital_beds_per100k',       # replaced by doctors_per100k (stronger link to premature mortality)

      # --- Join keys and metadata, not predictors ---
    'city_code',                   # ISGlobal join key only
    'year_env',                    # documents ISGlobal data year (2015), not a predictor

    # --- Data quality issues ---
    'one_person_hh_pct',           # 67% coverage, missing in many otherwise complete cities, confusing interpretation
    'murders_violent_per100k',     # cross-country comparability issues (different national definitions),
                                   # Netherlands anomaly, systematic country gaps
], errors='ignore')

In [34]:
# 6.2 Filter to cities with mortality data 2015-2019 (target period - good coverage, before covid)

mort_period = master_noise_pol[
    (master_noise_pol['year'].isin([2015, 2016, 2017, 2018, 2019])) &
    (master_noise_pol['deaths_under65_rate'].notna())
]

# All numeric columns to average (excluding year)
numeric_cols = mort_period.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'year']

# Average per city
export_df = mort_period.groupby('City_code')[numeric_cols].mean().reset_index()

# Add city name and country back
city_info = mort_period[['City_code', 'city', 'Country', 'NUTS2', 'NUTS3']].drop_duplicates(subset='City_code')
export_df = export_df.merge(city_info, on='City_code', how='left')

# Reorder columns - identifiers first, then target, then predictors
id_cols = ['City_code', 'city', 'Country', 'NUTS2', 'NUTS3']
target_col = ['deaths_under65_rate']
predictor_cols = [c for c in export_df.columns if c not in id_cols + target_col]
export_df = export_df[id_cols + target_col + predictor_cols]

noise_cols = ['total_lden_pct_55', 'total_lnight_pct_55']

n_missing = export_df[predictor_cols].isna().sum(axis=1)
noise_missing = export_df[noise_cols].isna().sum(axis=1)
other_missing = n_missing - noise_missing

print(f'Cities before dropping: {len(export_df)}')
print(f'Cities complete (0 missing): {(n_missing == 0).sum()}')
print(f'Cities missing only noise (noise NaN, rest complete): {((noise_missing > 0) & (other_missing == 0)).sum()}')
print(f'Cities with other gaps: {(other_missing > 0).sum()}')

# Keep only complete or missing only noise
export_df = export_df[(other_missing == 0)].reset_index(drop=True)
print(f'Cities after dropping: {len(export_df)}')

Cities before dropping: 788
Cities complete (0 missing): 230
Cities missing only noise (noise NaN, rest complete): 272
Cities with other gaps: 286
Cities after dropping: 502


In [ ]:
# 6.3 export final master file covering period 2015-2019
export_df.to_excel('DATA/OUTPUT/master_2015_2019.xlsx', index=False)
print('Exported to DATA/OUTPUT/master_2015_2019.xlsx')

## 7. Attempting to add a validation dataset to the study covering 2022-2024

In [ ]:
# Drop noise and pop_density from validation — not needed for model prediction
drop_val_cols = ['total_lnight_pct_55', 'total_lden_pct_55', 'pop_density']

mort_period_val = master_noise_pol[
    (master_noise_pol['year'].isin([2022, 2023, 2024])) &
    (master_noise_pol['deaths_under65_rate'].notna())
]

numeric_cols = mort_period_val.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'year']

export_val = mort_period_val.groupby('City_code')[numeric_cols].mean().reset_index()

city_info_val = mort_period_val[['City_code', 'city', 'Country', 'NUTS2', 'NUTS3']].drop_duplicates(subset='City_code')
export_val = export_val.merge(city_info_val, on='City_code', how='left')

# Drop noise and pop_density
export_val = export_val.drop(columns=drop_val_cols, errors='ignore')

id_cols = ['City_code', 'city', 'Country', 'NUTS2', 'NUTS3']
target_col = ['deaths_under65_rate']
predictor_cols = [c for c in export_val.columns if c not in id_cols + target_col]

export_val = export_val[id_cols + target_col + predictor_cols]

n_missing = export_val[predictor_cols].isna().sum(axis=1)
print(f'Cities before dropping: {len(export_val)}')
print(f'Cities complete (0 missing): {(n_missing == 0).sum()}')
print(f'Cities with gaps: {(n_missing > 0).sum()}')
print(f'\nMissing per variable:')
print(export_val[predictor_cols].isna().sum().sort_values(ascending=False))

export_val = export_val[n_missing == 0].reset_index(drop=True)
print(f'\nCities after dropping: {len(export_val)}')

**Unfortunately not enough data to validate the model on 2022-2024 dataset**